# ArcLoom L0 — First Hardware Test
## PYNQ-Z2 / Zynq-7020

This notebook:
1. Verifies the PYNQ board is alive
2. Loads test data (stock prices or any time series)
3. Computes L0 SEV in Python (reference implementation)
4. Eventually: loads the ArcLoom L0 hardware overlay and compares

**Step 1: Just prove the board works and we can send/receive data.**

In [ ]:
# Cell 1: Board check
from pynq import Overlay, PL
import platform
import numpy as np

print(f"Platform: {platform.machine()}")
print(f"PYNQ version: {pynq.__version__}" if hasattr(pynq, '__version__') else 'PYNQ loaded')
print(f"PL timestamp: {PL.timestamp}")
print(f"Board is ALIVE")

In [ ]:
# Cell 2: L0 reference implementation in Python
# This is the SAME math as the Verilog, but in software
# We'll use this to verify the hardware gives identical results

import numpy as np
import math

# UF-Spec v1.4.0 thresholds
ALPHA1 = 1.0
ALPHA2 = 1.0
ALPHA3 = 1.0
TAU_D = 0.20
SIGMA_MIN = 1e-6
DELTA_MIN = 1e-6
KAPPA_MIN = 1e-6
VARIANCE_WINDOW = 20

def compute_l0_sev(F_raw):
    """Compute L0 State Embedding Vector for a raw field series."""
    EPS = 1e-8
    F_norm = np.log(F_raw + EPS)
    n = len(F_norm)
    
    dF = np.zeros(n)
    sigma = np.zeros(n)
    kappa = np.zeros(n)
    N = np.zeros(n, dtype=int)
    D_t = np.zeros(n)
    gate_boundary = np.zeros(n, dtype=int)
    
    # dF
    if n > 1:
        dF[1:] = np.diff(F_norm)
    
    # sigma (windowed variance)
    w = max(1, VARIANCE_WINDOW)
    for i in range(n):
        start = max(0, i - w + 1)
        seg = F_norm[start:i+1]
        sigma[i] = float(np.var(seg)) if len(seg) > 1 else 0.0
    
    # kappa (curvature)
    if n > 2:
        for i in range(1, n-1):
            kappa[i] = abs(F_norm[i+1] - 2*F_norm[i] + F_norm[i-1])
    
    # N (negative space)
    for i in range(n):
        N[i] = 1 if (sigma[i] <= SIGMA_MIN and abs(dF[i]) <= DELTA_MIN and kappa[i] <= KAPPA_MIN) else 0
    
    # D(t) and gate boundary
    D_t = ALPHA1 * np.abs(dF) + ALPHA2 * sigma + ALPHA3 * kappa
    gate_boundary = (D_t > TAU_D).astype(int)
    
    return {
        'F_norm': F_norm,
        'dF': dF,
        'sigma': sigma,
        'kappa': kappa,
        'N': N,
        'D_t': D_t,
        'gate_boundary': gate_boundary,
        'n_gates': int(np.sum(gate_boundary)),
    }

# Test with synthetic data
np.random.seed(42)
test_data = 100 + np.cumsum(np.random.randn(200) * 0.5)
test_data = np.maximum(test_data, 1)  # keep positive

sev = compute_l0_sev(test_data)
print(f"Input: {len(test_data)} samples")
print(f"Gates detected: {sev['n_gates']}")
print(f"Negative space points: {np.sum(sev['N'])}")
print(f"D(t) range: [{sev['D_t'].min():.6f}, {sev['D_t'].max():.6f}]")
print(f"\nFirst 10 D(t) values:")
for i in range(10):
    print(f"  t={i}: F={test_data[i]:.2f} F_norm={sev['F_norm'][i]:.4f} dF={sev['dF'][i]:+.4f} D={sev['D_t'][i]:.4f} {'*GATE*' if sev['gate_boundary'][i] else ''}")

print(f"\nL0 reference implementation: WORKING")
print(f"This is what the FPGA will compute in O(1) per sample.")

In [ ]:
# Cell 3: Convert to 16.16 fixed-point (what the FPGA uses)

def to_fixed16_16(val):
    """Convert float to 16.16 fixed-point integer."""
    return int(val * 65536) & 0xFFFFFFFF

def from_fixed16_16(val):
    """Convert 16.16 fixed-point integer to float."""
    if val >= 0x80000000:  # negative
        val = val - 0x100000000
    return val / 65536.0

# Verify conversion
test_vals = [0.0, 1.0, -1.0, 0.20, 3.14159, -0.001, 100.5]
print("Fixed-point conversion test:")
for v in test_vals:
    fp = to_fixed16_16(v)
    back = from_fixed16_16(fp)
    err = abs(back - v)
    print(f"  {v:>10.5f} -> 0x{fp:08X} -> {back:>10.5f}  (err={err:.6f})")

# Convert L0 input data to fixed-point for FPGA
F_norm_fixed = np.array([to_fixed16_16(v) for v in sev['F_norm']], dtype=np.uint32)
print(f"\nConverted {len(F_norm_fixed)} samples to 16.16 fixed-point")
print(f"Ready to send to FPGA when overlay is loaded.")

In [ ]:
# Cell 4: When the hardware overlay is ready, this cell loads it
# and feeds data through the FPGA
#
# UNCOMMENT WHEN arcloom_l0.bit IS BUILT AND UPLOADED
#
# from pynq import Overlay, MMIO
# import time
#
# # Load the ArcLoom L0 overlay
# ol = Overlay('arcloom_l0.bit')
# print(f"Overlay loaded: {ol.bitfile_name}")
# print(f"IP blocks: {ol.ip_dict.keys()}")
#
# # Get the L0 boundary operator IP
# l0 = ol.arcloom_l0_sev_0
#
# # Feed data and read results
# results_hw = []
# for i, f_val in enumerate(F_norm_fixed):
#     l0.write(0x10, int(f_val))  # write F_norm_in
#     l0.write(0x00, 1)           # trigger valid_in
#     D_t = l0.read(0x20)         # read D_t_out
#     gate = l0.read(0x24)        # read gate_boundary
#     results_hw.append((from_fixed16_16(D_t), gate))
#
# # Compare HW vs SW
# print(f"HW vs SW comparison:")
# for i in range(min(10, len(results_hw))):
#     hw_D, hw_gate = results_hw[i]
#     sw_D = sev['D_t'][i]
#     sw_gate = sev['gate_boundary'][i]
#     match = 'OK' if (hw_gate == sw_gate) else 'MISMATCH'
#     print(f"  t={i}: HW_D={hw_D:.4f} SW_D={sw_D:.4f} HW_gate={hw_gate} SW_gate={sw_gate} {match}")

print("Cell 4: Hardware test placeholder — uncomment when overlay is built")